# HW10-11: CNN, transfer learning, сегментация (VOC)

Часть A: STL10 — C1…C4. Часть B: Pascal VOC **segmentation**, FCN-ResNet50 (V1 argmax, V2 median post).
Запуск из каталога `homeworks/HW10-11/`.

In [ ]:
import os, json, csv, random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, random_split
import torchvision
from torchvision import transforms, models
from torchvision.datasets import STL10, VOCSegmentation
from torchvision.models.segmentation import fcn_resnet50, FCN_ResNet50_Weights

# --- cwd: HW10-11 ---
ROOT = Path.cwd().resolve()
if ROOT.name != "HW10-11":
    cand = ROOT / "homeworks" / "HW10-11"
    if cand.is_dir():
        os.chdir(cand)
ART = Path("artifacts")
FIG = ART / "figures"
FIG.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

DATA_ROOT = "data"
DATASET_A = "STL10"
NUM_CLASSES = 10
BATCH = 32

# STL10: train 5000, test 8000 — val из train 80/20
base_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
aug_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(96, padding=8),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
imagenet_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
imagenet_aug_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_full = STL10(DATA_ROOT, split="train", download=True, transform=base_tf)
test_ds = STL10(DATA_ROOT, split="test", download=True, transform=base_tf)
n_val = int(0.2 * len(train_full))
n_tr = len(train_full) - n_val
g = torch.Generator().manual_seed(SEED)
train_base, val_base = random_split(train_full, [n_tr, n_val], generator=g)

train_full_aug = STL10(DATA_ROOT, split="train", download=True, transform=aug_tf)
train_aug = Subset(train_full_aug, train_base.indices)

test_loader = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=0)
val_loader_base = DataLoader(val_base, batch_size=BATCH, shuffle=False, num_workers=0)

def loaders_for_c12(aug_train: bool):
    tr = DataLoader(train_aug if aug_train else train_base, batch_size=BATCH, shuffle=True, num_workers=0)
    return tr, val_loader_base

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(256 * 12 * 12, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, NUM_CLASSES))

    def forward(self, x):
        return self.fc(self.f(x))

def train_one_epoch(model, loader, crit, opt):
    model.train()
    tot_l = tot = cor = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        logits = model(x)
        loss = crit(logits, y)
        loss.backward()
        opt.step()
        tot_l += loss.item() * x.size(0)
        tot += x.size(0)
        cor += (logits.argmax(1) == y).float().sum().item()
    return tot_l / tot, cor / tot

@torch.no_grad()
def evaluate(model, loader, crit):
    model.eval()
    tot_l = tot = cor = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = crit(logits, y)
        tot_l += loss.item() * x.size(0)
        tot += x.size(0)
        cor += (logits.argmax(1) == y).float().sum().item()
    return tot_l / tot, cor / tot

def train_cnn_epochs(model, train_loader, epochs, tag):
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    hist = {k: [] for k in ("tl", "ta", "vl", "va")}
    best_va, best_state = 0.0, None
    for ep in range(epochs):
        tl, ta = train_one_epoch(model, train_loader, crit, opt)
        vl, va = evaluate(model, val_loader_base, crit)
        for k, v in zip(hist.keys(), (tl, ta, vl, va)):
            hist[k].append(v)
        if va > best_va:
            best_va = va
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if (ep + 1) % max(1, epochs // 3) == 0:
            print(tag, "ep", ep + 1, "val_acc", va)
    if best_state is not None:
        model.load_state_dict(best_state)
    return best_va, min(hist["vl"]), hist

EPOCHS_CNN = 12

# C1
print("C1 simple CNN, no aug")
tr, _ = loaders_for_c12(False)
m1 = SmallCNN().to(device)
c1_acc, c1_loss, h1 = train_cnn_epochs(m1, tr, EPOCHS_CNN, "C1")

# C2
print("C2 simple CNN + aug")
tr2, _ = loaders_for_c12(True)
m2 = SmallCNN().to(device)
c2_acc, c2_loss, h2 = train_cnn_epochs(m2, tr2, EPOCHS_CNN, "C2")

# --- ResNet18: train/val/test с imagenet transforms ---
train_rn = STL10(DATA_ROOT, split="train", download=True, transform=imagenet_aug_tf)
val_rn = STL10(DATA_ROOT, split="train", download=True, transform=imagenet_tf)
test_rn = STL10(DATA_ROOT, split="test", download=True, transform=imagenet_tf)
train_rn_s = Subset(train_rn, train_base.indices)
val_rn_s = Subset(val_rn, val_base.indices)
tr_rn = DataLoader(train_rn_s, batch_size=BATCH, shuffle=True, num_workers=0)
val_rn_loader = DataLoader(val_rn_s, batch_size=BATCH, shuffle=False, num_workers=0)
test_rn_loader = DataLoader(test_rn, batch_size=BATCH, shuffle=False, num_workers=0)

weights = models.ResNet18_Weights.DEFAULT
def make_resnet():
    m = models.resnet18(weights=weights)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return m.to(device)

def train_resnet(model, freeze_backbone: bool, epochs, tag, lr=1e-3):
    crit = nn.CrossEntropyLoss()
    if freeze_backbone:
        for p in model.parameters():
            p.requires_grad = False
        for p in model.fc.parameters():
            p.requires_grad = True
        opt = torch.optim.Adam(model.fc.parameters(), lr=lr)
    else:
        for p in model.parameters():
            p.requires_grad = False
        for n, p in model.named_parameters():
            if n.startswith("layer4") or n.startswith("fc"):
                p.requires_grad = True
        opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr * 0.1)
    hist = {k: [] for k in ("tl", "ta", "vl", "va")}
    best_va, best_state = 0.0, None
    for ep in range(epochs):
        tl, ta = train_one_epoch(model, tr_rn, crit, opt)
        vl, va = evaluate(model, val_rn_loader, crit)
        for k, v in zip(hist.keys(), (tl, ta, vl, va)):
            hist[k].append(v)
        if va > best_va:
            best_va = va
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(tag, "ep", ep + 1, "val_acc", va)
    if best_state:
        model.load_state_dict(best_state)
    return best_va, min(hist["vl"]), hist

EPOCHS_RN = 8
print("C3 ResNet18 head only")
m3 = make_resnet()
c3_acc, c3_loss, h3 = train_resnet(m3, True, EPOCHS_RN, "C3")

print("C4 ResNet18 partial finetune layer4+fc")
m4 = make_resnet()
c4_acc, c4_loss, h4 = train_resnet(m4, False, EPOCHS_RN, "C4")

best_tag = max(
    [("C1", c1_acc), ("C2", c2_acc), ("C3", c3_acc), ("C4", c4_acc)],
    key=lambda x: x[1],
)[0]
models_map = {"C1": (m1, "SmallCNN", h1), "C2": (m2, "SmallCNN", h2), "C3": (m3, "ResNet18 head", h3), "C4": (m4, "ResNet18 finetune", h4)}
best_model, best_name, best_hist = models_map[best_tag]

crit = nn.CrossEntropyLoss()
if best_tag in ("C1", "C2"):
    test_loader_final = test_loader
else:
    best_model = best_model  # already resnet
    test_loader_final = test_rn_loader

@torch.no_grad()
def acc_on_loader(model, loader):
    model.eval()
    tot = cor = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        cor += (model(x).argmax(1) == y).sum().item()
        tot += y.size(0)
    return cor / tot

if best_tag in ("C1", "C2"):
    test_acc = acc_on_loader(best_model, test_loader)
else:
    test_acc = acc_on_loader(best_model, test_rn_loader)

print("Best by val:", best_tag, "test_acc", test_acc)

torch.save(best_model.state_dict(), ART / "best_classifier.pt")
with open(ART / "best_classifier_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "dataset": DATASET_A,
        "seed": SEED,
        "best_experiment": best_tag,
        "best_val_accuracy": float(max(c1_acc, c2_acc, c3_acc, c4_acc)),
        "test_accuracy": float(test_acc),
        "model": best_name,
    }, f, indent=2)

# figures
fig, ax = plt.subplots()
ax.bar(["C1", "C2", "C3", "C4"], [c1_acc, c2_acc, c3_acc, c4_acc])
ax.set_ylabel("best val acc (reload per run)")
ax.set_title("STL10 classifiers")
plt.tight_layout()
plt.savefig(FIG / "classification_compare.png", dpi=120)
plt.close()

# curves best (use hist of winner)
h_best = best_hist
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ep = range(1, len(h_best["tl"]) + 1)
ax[0].plot(ep, h_best["tl"], label="train")
ax[0].plot(ep, h_best["vl"], label="val")
ax[0].set_title("Loss " + best_tag)
ax[0].legend()
ax[1].plot(ep, h_best["ta"], label="train")
ax[1].plot(ep, h_best["va"], label="val")
ax[1].set_title("Acc " + best_tag)
ax[1].legend()
plt.tight_layout()
plt.savefig(FIG / "classification_curves_best.png", dpi=120)
plt.close()

# aug preview
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
it = iter(DataLoader(train_aug, batch_size=1, shuffle=True))
for i in range(8):
    x, _ = next(it)
    axes.flat[i].imshow(x[0].permute(1, 2, 0).numpy() * 0.5 + 0.5)
    axes.flat[i].axis("off")
plt.suptitle("Augmented STL10 samples")
plt.tight_layout()
plt.savefig(FIG / "augmentations_preview.png", dpi=120)
plt.close()

# ========= Part B: VOC Segmentation (FCN-ResNet50, те же классы PASCAL) =========
seg_w = FCN_ResNet50_Weights.DEFAULT
seg_model = fcn_resnet50(weights=seg_w).to(device)
seg_model.eval()
seg_pre = seg_w.transforms()

voc_root = Path(DATA_ROOT) / "voc"
voc_val = VOCSegmentation(str(voc_root), year="2012", image_set="val", download=True)


def miou_np(pred, tgt):
    pred = pred.reshape(-1)
    tgt = tgt.reshape(-1)
    m = tgt != 255
    pred, tgt = pred[m], tgt[m]
    ious = []
    for c in range(21):
        p = pred == c
        t = tgt == c
        u = (p | t).sum()
        if u > 0:
            ious.append((p & t).sum() / u)
    return float(np.mean(ious)) if ious else 0.0


def pixel_pr(pred, tgt):
    m = tgt != 255
    pred, tgt = pred[m], tgt[m]
    return float((pred == tgt).mean())


def eval_seg(n_img=30, postprocess=None):
    rng = np.random.default_rng(SEED)
    ix = rng.choice(len(voc_val), size=min(n_img, len(voc_val)), replace=False)
    m_ious, pxs = [], []
    for i in ix:
        img, mask = voc_val[int(i)]
        mask = np.array(mask, dtype=np.int64)
        inp = seg_pre(img).unsqueeze(0).to(device)
        with torch.no_grad():
            out = seg_model(inp)["out"]
        pred = torch.argmax(out, dim=1).squeeze(0).cpu().numpy()
        if postprocess is not None:
            pred = postprocess(pred)
        m_ious.append(miou_np(pred, mask))
        pxs.append(pixel_pr(pred, mask))
    return float(np.mean(m_ious)), float(np.mean(pxs))


def post_median(pred):
    from scipy.ndimage import median_filter
    return median_filter(pred, size=5)


m1, px1 = eval_seg(postprocess=None)
m2, px2 = eval_seg(postprocess=post_median)
print("V1 argmax mIoU", m1, "pixel_acc", px1)
print("V2 median-filter mIoU", m2, "pixel_acc", px2)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for k in range(4):
    img, mask = voc_val[k]
    mask = np.array(mask)
    inp = seg_pre(img).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = torch.argmax(seg_model(inp)["out"], dim=1).squeeze(0).cpu().numpy()
    axes[0, k].imshow(np.array(img))
    axes[0, k].set_title("image")
    axes[0, k].axis("off")
    axes[1, k].imshow(pred, vmin=0, vmax=20)
    axes[1, k].set_title("pred")
    axes[1, k].axis("off")
plt.tight_layout()
plt.savefig(FIG / "segmentation_examples.png", dpi=110)
plt.close()

fig, ax = plt.subplots()
ax.bar(["V1 mIoU", "V2 mIoU", "V1 pix", "V2 pix"], [m1, m2, px1, px2])
ax.set_ylim(0, 1)
plt.title("Segmentation metrics (val subset)")
plt.tight_layout()
plt.savefig(FIG / "segmentation_metrics.png", dpi=120)
plt.close()

rows = [
    {"experiment_id": "C1", "task": "classification", "dataset": DATASET_A, "seed": SEED,
     "model_summary": "SmallCNN no aug", "optimizer": "Adam", "lr": 1e-3, "epochs_trained": EPOCHS_CNN,
     "best_val_accuracy": c1_acc, "test_accuracy": float(test_acc) if best_tag == "C1" else "", "precision": "", "recall": "", "mean_iou": "", "notes": ""},
    {"experiment_id": "C2", "task": "classification", "dataset": DATASET_A, "seed": SEED,
     "model_summary": "SmallCNN aug", "optimizer": "Adam", "lr": 1e-3, "epochs_trained": EPOCHS_CNN,
     "best_val_accuracy": c2_acc, "test_accuracy": float(test_acc) if best_tag == "C2" else "", "precision": "", "recall": "", "mean_iou": "", "notes": ""},
    {"experiment_id": "C3", "task": "classification", "dataset": DATASET_A, "seed": SEED,
     "model_summary": "ResNet18 head", "optimizer": "Adam", "lr": 1e-3, "epochs_trained": EPOCHS_RN,
     "best_val_accuracy": c3_acc, "test_accuracy": float(test_acc) if best_tag == "C3" else "", "precision": "", "recall": "", "mean_iou": "", "notes": ""},
    {"experiment_id": "C4", "task": "classification", "dataset": DATASET_A, "seed": SEED,
     "model_summary": "ResNet18 layer4+fc", "optimizer": "Adam", "lr": 1e-4, "epochs_trained": EPOCHS_RN,
     "best_val_accuracy": c4_acc, "test_accuracy": float(test_acc) if best_tag == "C4" else "", "precision": "", "recall": "", "mean_iou": "", "notes": "test только для лучшего по val"},
    {"experiment_id": "V1", "task": "segmentation", "dataset": "VOC2012-seg", "seed": SEED,
     "model_summary": "FCN_ResNet50 VOC weights", "optimizer": "", "lr": "", "epochs_trained": 0,
     "best_val_accuracy": "", "test_accuracy": "", "precision": px1, "recall": "", "mean_iou": m1, "notes": "argmax"},
    {"experiment_id": "V2", "task": "segmentation", "dataset": "VOC2012-seg", "seed": SEED,
     "model_summary": "FCN_ResNet50 VOC weights", "optimizer": "", "lr": "", "epochs_trained": 0,
     "best_val_accuracy": "", "test_accuracy": "", "precision": px2, "recall": "", "mean_iou": m2, "notes": "median filter 5x5"},
]
with open(ART / "runs.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    w.writerows(rows)
print("Done HW10-11")